# 🧵 15 · Classical Feature-Based Bioimage Classification
### From pixels to a decision: texture, shape, and distance for classical classifiers

**Part D · Classifying bioimages** · Module 15 of 37 (core) · builds on 11, 13, 14

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/15_classical_feature_based_classification.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (free Colab is plenty) |
| **Core frameworks** | NumPy, SciPy, scikit-learn |
| **Prerequisites** | Modules 11 (filtering), 13 (keypoints), 14 (edges) |
| **Estimated time** | 70–90 minutes |
| **02-740 topics covered** | feature extraction, GLCM/Haralick texture, geometric/morphological/statistical features, Euclidean/z-score/Mahalanobis distance, feature selection, decision trees, k-NN, SVM and kernels |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every nucleus image below is generated by `course_toolkit.simulate_nuclei_image`; the shape label that the classifiers try to recover is recorded by the simulator itself, so we always know the right answer.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why raw pixel intensities make poor classification features, and what a *feature* is instead.
- Compute grey-level histogram statistics and grey-level co-occurrence matrix (GLCM) texture features,
  and name what each Haralick statistic (contrast, energy, homogeneity, correlation) is sensitive to.
- Compute simple geometric/morphological features (area, perimeter, compactness, aspect ratio) of a
  segmented object.
- Compute and compare Euclidean, z-score and Mahalanobis distance between feature vectors, and explain
  why Mahalanobis is the only one of the three that accounts for correlated features.
- Train and compare three classical classifiers — a decision tree, k-nearest neighbours, and a support
  vector machine with a kernel — on a shared feature matrix, and read a confusion matrix.

## 🗺️ Contents

1. From pixels to a decision: the classical classification pipeline
2. Texture features: histograms and the grey-level co-occurrence matrix
3. Geometric and morphological features
4. Comparing feature vectors: Euclidean, z-score and Mahalanobis distance
5. Feature selection in one sentence
6. Classical classifiers: decision trees, k-nearest neighbours, and the support vector machine

## 🧫 The biological question

A microscopy core facility hands you ten thousand images of nuclei from a screen and asks: *which of
these nuclei are round, which are dividing, which look irregular (a hint of stress or disease)?* Nobody
has time to look at ten thousand images by eye. You need a classifier — but a classifier needs *features*,
numbers computed from each image that capture what a human eye would notice (shape, texture, brightness
pattern) well enough for a simple algorithm to tell the classes apart.

This notebook builds that pipeline end to end, on simulated nuclei where we — unlike a real screen — know
the true shape label of every cell, so every step can be checked against ground truth.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 70
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

Run this once — it is an exact copy of the shared `course_toolkit.py` functions this notebook uses, pasted here so the notebook runs on its own in Colab.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def glcm_features(image: np.ndarray, levels: int = 8, dx: int = 1, dy: int = 0):
    """Haralick-style texture features from a grey-level co-occurrence matrix (GLCM).

    Quantises ``image`` to ``levels`` grey levels, builds the co-occurrence matrix for the
    offset (dy, dx), and returns a dict of contrast, energy (angular second moment),
    homogeneity and correlation.
    """
    img = image - image.min()
    img = img / (img.max() + 1e-9)
    q = np.clip((img * (levels - 1)).round().astype(int), 0, levels - 1)
    h, w = q.shape
    glcm = np.zeros((levels, levels))
    y0, y1 = max(0, -dy), h - max(0, dy)
    x0, x1 = max(0, -dx), w - max(0, dx)
    a = q[y0:y1, x0:x1]
    b = q[y0 + dy:y1 + dy, x0 + dx:x1 + dx]
    for i, j in zip(a.ravel(), b.ravel()):
        glcm[i, j] += 1
    glcm /= glcm.sum() + 1e-9
    i_idx, j_idx = np.meshgrid(np.arange(levels), np.arange(levels), indexing="ij")
    mu_i, mu_j = (glcm * i_idx).sum(), (glcm * j_idx).sum()
    sig_i = np.sqrt((glcm * (i_idx - mu_i) ** 2).sum()) + 1e-9
    sig_j = np.sqrt((glcm * (j_idx - mu_j) ** 2).sum()) + 1e-9
    contrast = (glcm * (i_idx - j_idx) ** 2).sum()
    energy = (glcm ** 2).sum()
    homogeneity = (glcm / (1 + np.abs(i_idx - j_idx))).sum()
    correlation = ((glcm * (i_idx - mu_i) * (j_idx - mu_j)).sum()) / (sig_i * sig_j)
    return dict(contrast=contrast, energy=energy, homogeneity=homogeneity, correlation=correlation)


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def mahalanobis_distance(x: np.ndarray, mean: np.ndarray, cov_inv: np.ndarray) -> float:
    """Mahalanobis distance of a feature vector ``x`` from a class with the given mean and the
    INVERSE of its covariance matrix (pass ``np.linalg.pinv(cov)`` once, reuse for every point)."""
    d = x - mean
    return float(np.sqrt(d @ cov_inv @ d))


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. From pixels to a decision: the classical classification pipeline

A classical (pre-deep-learning) bioimage classifier is a pipeline, not a single algorithm:

1. **Pre-process** the raw image (denoise, correct illumination — Module 11).
2. **Detect/segment** the objects of interest, so you know *which pixels* belong to each cell
   (Modules 02, 18 cover this in depth; here we use the simulator's ground-truth masks directly).
3. **Extract features**: turn each object's pixels into a short vector of numbers.
4. **Select features**: drop the ones that do not help, to avoid confusing the classifier.
5. **Train and classify**: fit a classifier on labelled examples (the *training set*), then use it to
   predict labels for new, unseen images (the *testing set*).

*In words:* step 3 is the crux of this notebook. Raw pixel intensities are a poor choice of feature
because they encode absolute position and orientation ("this bright pixel is at row 54") rather than the
translation- and rotation-invariant properties (texture, shape, intensity statistics) that actually define
a biological class. Two cells of the identical shape, one rotated 90°, should get the same label — but
their raw pixel vectors can look completely different.

A classifier is only as good as its ability to **generalize**: fitting the training data perfectly (by
memorising its noise) is *overfitting* and usually makes test performance worse, not better; fitting it
too loosely is *underfitting*. We will watch for both as we go.

In [ ]:
# ── 1.1 Build a small labelled gallery of simulated nuclei ──
gallery = [simulate_nuclei_image(n_cells=1, size=96, shape_classes=(c,), seed=100 + i)
           for i, c in enumerate(SHAPE_CLASSES) for _ in range(2)]
images = [g["image"] for g in gallery]
labels = [g["shape_labels"][0] for g in gallery]
show_image_grid(images, titles=labels, ncols=4)
plt.show()
print("True shape labels:", labels)

🔍 **What just happened?** We simulated two example nuclei of every shape
class in `SHAPE_CLASSES`. The simulator hands back the true `shape_labels` alongside the pixels — our
"ground truth" for the rest of the notebook. A real experiment would need manual annotation (slow,
subjective) or an experimental design that fixes the label (fast, objective) to get this same information;
simulated data lets us skip straight to testing the pipeline.

### 📘 Lecture notes — why pixels alone are not features

Two images of the same cell type, shifted by ten pixels or rotated by 40°, are extremely different
*vectors* of raw intensities (every entry can change) yet obviously the same biological object. A useful
feature should instead be approximately invariant to the irrelevant nuisance variables (position,
orientation, often also scale) and sensitive to the variables that define the class (shape, texture). The
rest of this notebook builds exactly such features.

## 2. Texture features: histograms and the grey-level co-occurrence matrix

Bioimage features are usually grouped into six families: **texture**, **geometrical**, **statistical**,
**intensity**, **edge** and **morphological**. This section covers texture — "the spatial arrangement of
intensities" — which comes in two flavours.

**First-order (histogram) texture** ignores spatial arrangement and just describes the distribution of
grey values: the mean, standard deviation, skewness (asymmetry), kurtosis ("tailedness") and entropy
(information content / disorder) of the pixel-value histogram. A patch of uniform cytoplasm has low
entropy; a patch littered with bright granules has high entropy.

**Second-order (co-occurrence) texture** looks at *pairs* of pixel values a fixed offset apart. The
**grey-level co-occurrence matrix (GLCM)** $P(g_1, g_2)$ counts, for every pair of grey levels
$(g_1, g_2)$, how often a pixel with value $g_1$ has a neighbour at some fixed offset with value $g_2$
(Haralick et al., 1973). From the normalized GLCM, Haralick's texture features summarize the matrix's
shape, including:

- **Energy / angular second moment** $\sum_{g_1,g_2} P(g_1,g_2)^2$ — high when a few grey-level pairs
  dominate (a uniform, orderly texture); low when pairs are spread out (a busy texture).
- **Contrast** $\sum_{g_1,g_2} (g_1-g_2)^2\,P(g_1,g_2)$ — large local grey-level differences (a rough,
  high-contrast texture) push this up.
- **Homogeneity / inverse difference moment** $\sum_{g_1,g_2} \dfrac{P(g_1,g_2)}{1+|g_1-g_2|}$ — large
  when most mass sits near the GLCM diagonal, i.e. neighbouring pixels usually have similar values.
- **Correlation** — the Pearson correlation of $g_1$ and $g_2$ under $P$: how linearly predictable a
  pixel's value is from its neighbour's.

*In words:* a perfectly smooth gradient has high homogeneity and low contrast; a salt-and-pepper texture
has the opposite.

In [ ]:
# ── 2.1 Grey-level histogram features for two different textures ──
def histogram_features(image):
    """First-order texture features from an image's grey-level histogram."""
    x = image.ravel()
    mean, std = x.mean(), x.std() + 1e-9
    skew = np.mean(((x - mean) / std) ** 3)
    kurt = np.mean(((x - mean) / std) ** 4)
    hist, _ = np.histogram(x, bins=32, density=True)
    p = hist[hist > 0]
    entropy = -np.sum(p * np.log2(p)) * (x.max() - x.min()) / 32
    return dict(mean=mean, std=std, skewness=skew, kurtosis=kurt, entropy=entropy)

smooth = simulate_nuclei_image(n_cells=1, size=96, shape_classes=("round",), noise_sigma=0.01, seed=1)["image"]
noisy = simulate_nuclei_image(n_cells=1, size=96, shape_classes=("irregular",), noise_sigma=0.08, seed=2)["image"]
for name, img in [("low-noise round nucleus", smooth), ("noisier irregular nucleus", noisy)]:
    print(name, "->", {k: round(v, 4) for k, v in histogram_features(img).items()})

In [ ]:
# ── 2.2 GLCM / Haralick features at 0 degrees ──
feats_smooth = glcm_features(smooth, levels=8, dx=1, dy=0)
feats_noisy = glcm_features(noisy, levels=8, dx=1, dy=0)
print("Low-noise round nucleus  :", {k: round(v, 4) for k, v in feats_smooth.items()})
print("Noisier irregular nucleus:", {k: round(v, 4) for k, v in feats_noisy.items()})

✏️ **Your turn.** `glcm_features` takes a `dy, dx` offset (we used `dx=1, dy=0`,
i.e. "one pixel to the right"). In the lecture notes, GLCM is usually computed and *summed* over several
angles (0°, 45°, 90°, 135°) to make the feature rotation-invariant. Recompute `feats_smooth` with
`dy=1, dx=0` (straight down) instead, and again with `dy=1, dx=1` (diagonal) — do the four Haralick
numbers change much for a round, roughly isotropic nucleus? Would you expect them to change more for the
`"elongated"` shape class?

🧪 **Ground-truth check.** We *know* the noisy image has five times the
simulated noise of the smooth one (`noise_sigma=0.08` vs. `0.01`) — and indeed its histogram entropy and
GLCM contrast should both come out higher. If they did not, something in the feature code (or our
intuition about what these features measure) would be wrong — this is the value of working with
simulated, ground-truth data while developing a pipeline.

### 📘 Lecture notes — reading a GLCM

For an image with grey levels in $\{0, 1, 2, 3\}$ and the offset "one pixel to the right," the GLCM
entry $P(1, 2)$ counts how many times a pixel valued $1$ has a right-hand neighbour valued $2$,
*normalized* so the whole matrix sums to 1. GLCM is traditionally computed at several offsets (the four
angles above) and summed, because a single offset only probes texture along one direction — a useful
texture descriptor for, say, striped actin fibers should not depend on which way the fibers happen to be
oriented under the microscope.

## 3. Geometric and morphological features

Once an object is segmented (its binary mask is known), a family of **geometric** features describes its
shape directly:

- **Area** — the pixel count inside the mask.
- **Perimeter** — the number of boundary pixels.
- **Compactness** $\dfrac{4\pi\,\text{area}}{\text{perimeter}^2}$ — equals 1.0 for a perfect circle and
  shrinks as the shape becomes more convoluted (this is also called the *circularity shape factor*).
- **Aspect ratio** — the bounding box's longer side over its shorter side; near 1 for round or square
  shapes, large for elongated ones.

**Morphological** features extend this family to objects defined by thresholding (number of objects,
their average size and variance, skeleton length and branch points), and **intensity** features (mean,
max intensity within the mask) and **edge** features (edge-direction homogeneity — do edges mostly point
the same way, as in parallel actin filaments, or every which way, as in a star-shaped tubulin pattern?)
round out the six feature families from Section 1.

In [ ]:
# ── 3.1 Geometric/morphological features across all shape classes ──
rows = []
for shape in SHAPE_CLASSES:
    for trial in range(6):
        sim = simulate_nuclei_image(n_cells=1, size=96, shape_classes=(shape,), seed=200 + trial)
        mask = sim["instance_mask"] > 0
        feats = shape_features(mask)
        feats["shape"] = shape
        rows.append(feats)

import collections
by_shape = collections.defaultdict(list)
for r in rows:
    by_shape[r["shape"]].append(r)
for shape, items in by_shape.items():
    comp = np.mean([it["compactness"] for it in items])
    ar = np.mean([it["aspect_ratio"] for it in items])
    print(f"{shape:10s}  mean compactness={comp:.3f}  mean aspect_ratio={ar:.3f}")

In [ ]:
# ── 3.2 Visualize the two geometric features that separate shapes best ──
fig, ax = plt.subplots(figsize=(5, 4))
for shape, items in by_shape.items():
    xs = [it["compactness"] for it in items]
    ys = [it["aspect_ratio"] for it in items]
    ax.scatter(xs, ys, label=shape)
ax.set_xlabel("compactness (1.0 = perfect circle)"); ax.set_ylabel("aspect ratio")
ax.legend(fontsize=8); ax.set_title("Geometric features by true shape class")
plt.show()

🔍 **What just happened?** Compactness and aspect ratio alone already
separate `"ring"` (low compactness, moderate aspect ratio) from `"round"` (high compactness, aspect ratio
≈ 1) from `"elongated"` (lower compactness, high aspect ratio) reasonably well — even before we bring in
texture. This is exactly why bioimage classification pipelines combine *several* feature families: no
single feature separates every class, but together they usually do.

## 4. Comparing feature vectors: Euclidean, z-score and Mahalanobis distance

Once every object is a feature vector, classification becomes a question of *distance*: is this new
vector closer to the typical "round" vector or the typical "elongated" vector?

For a **single** feature $j$ and two observations $i, m$:

$$d_{\text{univariate}} = |X_{i,j} - X_{m,j}|, \qquad
  d_{\text{z-score}} = \frac{|X_{i,j} - X_{m,j}|}{\sigma_j}$$

The z-score distance rescales by the feature's own standard deviation $\sigma_j$, so that a feature
measured in large units (area, in pixels²) does not automatically dominate a feature measured in small
units (compactness, roughly 0–1).

For **several** features at once, the natural generalization of ordinary distance is the
**multivariate Euclidean distance** $d = \sqrt{\sum_j (X_{i,j}-X_{m,j})^2}$ — but this still treats every
feature as independent and equally scaled, which is rarely true (area and perimeter, for instance, are
highly correlated). The **Mahalanobis distance**

$$d_{\text{Mahalanobis}}(\mathbf{x}, \boldsymbol\mu) =
  \sqrt{(\mathbf{x}-\boldsymbol\mu)^\top \boldsymbol\Sigma^{-1} (\mathbf{x}-\boldsymbol\mu)}$$

fixes both problems at once by dividing out the *full covariance matrix* $\boldsymbol\Sigma$ of the
reference class, not just each feature's own variance. It is unitless, scale-invariant, and automatically
down-weights distance along directions where the class already has a lot of natural spread.

In [ ]:
# ── 4.1 A small worked example: two cells, two features ──
# feature vector = (area, compactness); a reference class of five 'round' nuclei plus one new point
ref = np.array([[1850, 0.94], [1790, 0.91], [1920, 0.97], [1810, 0.93], [1870, 0.95]])
new_point = np.array([1830.0, 0.80])   # same area as the class, but noticeably less compact

mean = ref.mean(axis=0)
std = ref.std(axis=0)
cov_inv = np.linalg.pinv(np.cov(ref, rowvar=False))

euclid = float(np.linalg.norm(new_point - mean))
zscore = float(np.linalg.norm((new_point - mean) / std))
maha = mahalanobis_distance(new_point, mean, cov_inv)
print(f"Euclidean distance   : {euclid:8.3f}  (dominated by the area units, ~1800)")
print(f"Multivariate z-score : {zscore:8.3f}")
print(f"Mahalanobis distance : {maha:8.3f}")

✏️ **Your turn.** The Euclidean distance above is numerically huge (~tens)
purely because `area` is measured in the thousands of pixels while `compactness` lives in [0, 1] — one
feature silently swamps the other. Change `new_point`'s compactness to `0.95` (right in the middle of the
reference class) while keeping the area at `1830`. Which distance now correctly reports "very close to the
reference class," and which one still mostly reflects the (unremarkable) area difference?

### 📘 Lecture notes — why Mahalanobis uses the *inverse* covariance

Intuitively, $\boldsymbol\Sigma^{-1}$ "undoes" the reference class's own natural spread and correlation
structure before measuring distance: a direction in feature space where the class already varies a lot
(e.g., area naturally ranges over hundreds of pixels across normal cells) contributes *less* per unit of
raw difference, while a direction where the class is tightly clustered (e.g., compactness barely varies
for healthy round nuclei) contributes *more*. A point that is "two typical standard deviations away, after
accounting for how area and compactness naturally co-vary" is a much more meaningful notion of outlier
than raw Euclidean distance.

## 5. Feature selection in one sentence

Section 2–4 showed that even a handful of features (four GLCM statistics, four geometric statistics, plus
histogram statistics) already adds up quickly, and a real bioimage pipeline often computes dozens more.
*Having too many features can confuse a classifier* — some are redundant (perimeter and area are highly
correlated for round-ish objects), some are pure noise for the task at hand, and in high dimensions with
limited training data a classifier starts to fit noise instead of signal (we will meet this "curse of
dimensionality" properly in the next notebook). **Feature selection** methods — comparing per-class
feature distributions, information gain, or simply trying many feature subsets and keeping whichever gives
the best validation performance — trim the feature set down before training. **Feature extraction**
methods such as PCA do something related but different: instead of keeping a subset of the original
features, they build a smaller number of *new* features as combinations of the old ones (Module 17).

## 6. Classical classifiers: decision trees, k-nearest neighbours, and SVM

With features and distances in hand, three classical classifiers cover most of what you will see in
bioimage papers predating deep learning:

**Decision trees** repeatedly split the feature space along one feature at a time ("is area < 50?", then
"is compactness < 0.8?"...), producing a flowchart whose leaves are class labels. They are easy to
interpret but can overfit if grown too deep.

**k-nearest neighbours (k-NN)** has no training phase at all: to classify a new point, find its $k$
nearest neighbours (usually by Euclidean distance) in the training set and take a majority vote. Small
$k$ follows the local, noisy structure of the data closely (can overfit); large $k$ smooths more
aggressively (can underfit).

**Support vector machines (SVM)** look for the **maximum-margin separating hyperplane**: among all
hyperplanes $w\cdot x + b = 0$ that separate two classes, pick the one with the largest distance ("widest
corridor") to the nearest point of either class, because that is the boundary most likely to generalize
well to new points. When classes are not linearly separable, the **kernel trick** implicitly maps points
into a higher-dimensional space — via a kernel function such as the polynomial or Gaussian (RBF) kernel —
where a separating hyperplane *does* exist, without ever explicitly computing the high-dimensional
coordinates. For more than two classes, multiple SVMs are combined, either one-per-class
("one-against-all") or one-per-class-pair ("pairwise").

In [ ]:
# ── 6.1 Build a feature matrix for all six shape classes ──
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

def feature_vector(image, mask):
    g = glcm_features(image, levels=8)
    s = shape_features(mask)
    h = histogram_features(image)
    return [g["contrast"], g["energy"], g["homogeneity"], g["correlation"],
            s["compactness"], s["aspect_ratio"], h["entropy"], h["std"]]

X, y = [], []
for shape in SHAPE_CLASSES:
    for trial in range(40):
        sim = simulate_nuclei_image(n_cells=1, size=80, shape_classes=(shape,), seed=1000 + trial,
                                     noise_sigma=rng.uniform(0.02, 0.05))
        X.append(feature_vector(sim["image"], sim["instance_mask"] > 0))
        y.append(shape)
X, y = np.array(X), np.array(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
print("Feature matrix:", X.shape, "| classes:", sorted(set(y)))

In [ ]:
# ── 6.2 Train and compare the three classifiers ──
models = {
    "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=SEED),
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF kernel)": SVC(kernel="rbf", C=2.0, gamma="scale"),
}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name:25s} test accuracy = {acc:.2f}")

🔍 **What just happened?** All three classifiers see the exact same 8-number
feature vectors (GLCM texture + shape + histogram statistics) and the exact same train/test split — any
difference in accuracy comes purely from *how* each algorithm partitions feature space, not from the
features themselves. A linear SVM would likely do noticeably worse here than the RBF-kernel one, because
"round" vs. "ring" vs. "dividing" are not separable by a single straight boundary in this feature space —
exactly the non-linear situation the kernel trick is built for.

In [ ]:
# ── 6.3 A confusion matrix for the best-performing model ──
best_name, best_clf = "SVM (RBF kernel)", models["SVM (RBF kernel)"]
pred = best_clf.predict(X_test)
classes = sorted(set(y))
cm = confusion_matrix(y_test, pred, labels=classes)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{best_name} confusion matrix")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, shrink=0.8)
plt.tight_layout(); plt.show()

✏️ **Your turn.** Which pair of shape classes does the confusion matrix mix up
most often? Look back at `_draw_blob` in the toolkit (or just the gallery image from Section 1) — is there
a geometric reason those two shapes would be hard to separate using only compactness and aspect ratio?
Try adding `s["area"]` to `feature_vector` and see whether accuracy changes.

📌 **Key takeaways**

- Raw pixels are a poor classification feature; texture (GLCM/Haralick), geometric/morphological, and intensity features are translation- and (mostly) rotation-invariant summaries that a classifier can actually use.
- GLCM-based Haralick features (contrast, energy, homogeneity, correlation) summarize second-order texture — how pairs of neighbouring grey levels co-occur — not just the overall brightness histogram.
- Euclidean distance implicitly assumes all features are equally scaled and uncorrelated; z-score distance fixes the scaling; Mahalanobis distance fixes both scaling *and* correlation by using the full inverse covariance matrix.
- Decision trees, k-NN and SVM all classify the *same* feature vectors differently: trees split axis-by-axis, k-NN votes among nearby training points, and SVM finds a maximum-margin boundary (linear or, via a kernel, non-linear).

✅ **Checkpoint**

1. Why can two images of the identical nucleus, just rotated, have wildly different raw-pixel vectors but
   very similar GLCM/Haralick feature vectors?
   <details><summary>Answer</summary>GLCM counts co-occurring grey-level *pairs* at a fixed offset,
   which depends on the image's local texture statistics, not on where in the frame the object sits or
   exactly how it is rotated (especially once GLCM is summed over several offset angles) — whereas raw
   pixel intensities are tied to absolute pixel coordinates.</details>
2. A classifier gets 99% accuracy on the data it was trained on but only 60% on new test images. Is this
   over- or under-fitting, and what classical-ML knob would you turn first to fix it?
   <details><summary>Answer</summary>Overfitting — the model has memorised training-set idiosyncrasies
   rather than learning generalizable structure. First knobs to try: shrink a decision tree's max depth,
   increase k for k-NN, reduce an SVM's C (allow more margin violations), or simply collect more training
   data / fewer features.</details>
3. Why does Mahalanobis distance use the covariance matrix of the *reference class*, not of the whole
   dataset?
   <details><summary>Answer</summary>Because the question is "how unusual is this point *for this
   particular class*" — a feature's natural spread can differ from class to class (e.g. dividing nuclei
   might naturally vary more in area than round ones), and using the wrong class's covariance would
   mis-scale the distance.</details>

🧠 **Questions & discussion**

- The decision-tree classifier is the most *interpretable* of the three (you can read off its split rules), but SVM with an RBF kernel is usually more *accurate*. In a clinical pathology setting, which property would you prioritize, and why might regulators care?
- Our feature vector mixes texture, shape and intensity statistics computed at very different numeric scales. We did not z-score or Mahalanobis-normalize the inputs before handing them to scikit-learn's classifiers — would you expect that to matter more for k-NN/SVM or for the decision tree, and why?
- Real microscopy datasets are rarely perfectly balanced across classes (far more healthy cells than diseased ones, say). Which of accuracy, precision, recall, and F-measure would stay most informative under severe class imbalance?

🚀 **Bonus — going further**

- Add `n_fold` cross-validation (`sklearn.model_selection.cross_val_score`) instead of a single train/test split, and report mean ± standard deviation accuracy for each model.
- Implement a simple one-against-all multi-class SVM manually (one binary SVM per class, predict the class whose SVM gives the largest margin) and compare its accuracy to scikit-learn's built-in multi-class handling.
- Real texture analysis of actin vs. tubulin cytoskeleton images uses *edge-direction homogeneity* (are detected edges mostly parallel, as in actin fibers, or radially symmetric, as in a tubulin aster?) — after Module 14's Sobel gradients, try adding this as a ninth feature.

📝 **Lab note**

```text
Module 15 — Classical feature-based classification
Date: ____________
Feature families used: texture (GLCM) ___  geometric ___  intensity/histogram ___
Classifier accuracies:  tree ____  k-NN ____  SVM(RBF) ____
Most-confused class pair: ____________  why: ____________
One feature I would add next, and why: ____________
```

📚 **References**

- Haralick, R. M., Shanmugam, K. & Dinstein, I. (1973). Textural features for image classification. *IEEE Transactions on Systems, Man, and Cybernetics*, SMC-3(6), 610–621.
- Cortes, C. & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, 20, 273–297.
- Cover, T. & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, 13(1), 21–27.
- Breiman, L., Friedman, J., Olshen, R. & Stone, C. (1984). *Classification and Regression Trees*. Wadsworth.
- Markiewicz, T., Osowski, S., Mariańska, B. & Moszczyński, Ł. (2005). Automatic recognition of the blood cells of myelogenous leukemia using SVM. *IJCNN 2005*.
- Rodellar, J., Alférez, S., Acevedo, A., Molina, A. & Merino, A. (2018). Image processing and machine learning in the morphological analysis of blood cells. *International Journal of Laboratory Hematology*, 40(S1), 46–53.
- Mahalanobis, P. C. (1936). On the generalised distance in statistics. *Proceedings of the National Institute of Sciences of India*, 2(1), 49–55.

➡️ **Next.** Module 16 swings from hand-crafted features to features that are *learned*: artificial neurons, backpropagation, and convolutional neural networks, applied to the same kind of shape-classification problem.